### OpinionQA preprocess

In [ ]:
import json
from collections import defaultdict

In [ ]:
wave_year_map = {
    26: 2017,
    27: 2017,
    29: 2017,
    32: 2018,
    34: 2018,
    36: 2018,
    41: 2018,
    42: 2019,
    43: 2019,
    45: 2019,
    49: 2019,
    50: 2019,
    54: 2019,
    82: 2021,
    92: 2021
}


### OpinionQA preprocess

In [ ]:
wave='29'  # OpinionQA wave to preprocess, e.g. 26 or 29
data_dir=f'./dataset/opinionqa/American_Trends_Panel_W{wave}/opinionqa_W{wave}_per_user_with_gt.json'

In [ ]:
with open(data_dir) as f:
    data=json.load(f)

In [ ]:
features=defaultdict(list)
for d in data:
    for k,v in d['feature'].items():
        features[k].append(v)

In [ ]:
for k,v in features.items():
    features[k]=list(set(features[k]))

In [ ]:
import json
from collections import defaultdict
import string
from typing import List, Dict, Any, Optional

# Define all possible demographic values
DEMOGRAPHIC_VALUES = {
    'CREGION': ['Midwest', 'Northeast', 'South', 'West'],
    'AGE': ['', '18-29', '65+', 'Refused', '50-64', '30-49'],
    'SEX': ['In some other way', 'Female', 'Male', 'Refused'],
    'EDUCATION': ['Some college, no degree',
                  'Refused',
                  "Associate's degree",
                  'College graduate/some postgrad',
                  'Less than high school',
                  'Postgraduate',
                  'High school graduate'],
    'MARITAL': ['Living with a partner',
                '',
                'Refused',
                'Married',
                'Widowed',
                'Divorced',
                'Separated',
                'Never been married'],
    'RELIG': ['',
              'Mormon',
              'Atheist',
              'Refused',
              'Jewish',
              'Hindu',
              'Roman Catholic',
              'Unitarian',
              'Christian',
              'Nothing in particular',
              'Other',
              'Buddhist',
              'Agnostic',
              'Protestant',
              'Muslim',
              'Orthodox'],
    'RELIGATTEND': ['',
                    'A few times a year',
                    'Never',
                    'Refused',
                    'Once or twice a month',
                    'Once a week',
                    'More than once a week',
                    'Seldom'],
    'POLPARTY': ['',
                 'Republican',
                 'Refused',
                 'Other',
                 'Democrat',
                 'Independent'],
    'INCOME': ['',
               'Refused',
               '$100,000 or more',
               '$75,000-$100,000',
               'Less than $30,000',
               '$50,000-$75,000',
               '$30,000-$50,000'],
    'POLIDEOLOGY': ['',
                    'Conservative',
                    'Very liberal',
                    'Refused',
                    'Moderate',
                    'Liberal',
                    'Very conservative'],
    'RACE': ['Hispanic',
             'Refused',
             'Mixed Race',
             'White',
             'Black',
             'Other',
             'Asian']
}

# Define demographic description templates
DEMOGRAPHIC_DESCRIPTIONS = {
    'CREGION': {
        'Midwest': 'You are from the Midwest region.',
        'Northeast': 'You are from the Northeast region.',
        'South': 'You are from the South region.',
        'West': 'You are from the West region.'
    },
    'AGE': {
        '18-29': 'You are aged 18-29.',
        '30-49': 'You are aged 30-49.',
        '50-64': 'You are aged 50-64.',
        '65+': 'You are aged 65 or older.'
    },
    'SEX': {
        'Female': 'You are Female.',
        'Male': 'You are Male.',
        'In some other way': 'You identify your gender in some other way.'
    },
    'EDUCATION': {
        'Less than high school': 'Your highest education level is less than high school.',
        'High school graduate': 'Your highest education level is high school graduate.',
        'Some college, no degree': 'Your highest education level is some college, no degree.',
        "Associate's degree": "Your highest education level is an Associate's degree.",
        'College graduate/some postgrad': 'Your highest education level is college graduate or some postgraduate.',
        'Postgraduate': 'Your highest education level is postgraduate.'
    },
    'MARITAL': {
        'Married': 'You are married.',
        'Living with a partner': 'You are living with a partner.',
        'Widowed': 'You are widowed.',
        'Divorced': 'You are divorced.',
        'Separated': 'You are separated.',
        'Never been married': 'You have never been married.'
    },
    'RELIG': {
        'Protestant': 'Your religion is Protestant.',
        'Roman Catholic': 'Your religion is Roman Catholic.',
        'Mormon': 'Your religion is Mormon.',
        'Orthodox': 'Your religion is Orthodox Christian.',
        'Jewish': 'Your religion is Jewish.',
        'Muslim': 'Your religion is Muslim.',
        'Buddhist': 'Your religion is Buddhist.',
        'Hindu': 'Your religion is Hindu.',
        'Atheist': 'You are an Atheist.',
        'Agnostic': 'You are Agnostic.',
        'Nothing in particular': 'You have no particular religion.',
        'Unitarian': 'Your religion is Unitarian.',
        'Christian': 'Your religion is Christian.',
        'Other': 'Your religion is other.'
    },
    'RELIGATTEND': {
        'More than once a week': 'You attend religious services more than once a week.',
        'Once a week': 'You attend religious services once a week.',
        'Once or twice a month': 'You attend religious services once or twice a month.',
        'A few times a year': 'You attend religious services a few times a year.',
        'Seldom': 'You seldom attend religious services.',
        'Never': 'You never attend religious services.'
    },
    'POLPARTY': {
        'Republican': 'You identify as a Republican.',
        'Democrat': 'You identify as a Democrat.',
        'Independent': 'You identify as an Independent.',
        'Other': 'You identify with another political party.'
    },
    'INCOME': {
        'Less than $30,000': 'Your household income is less than $30,000.',
        '$30,000-$50,000': 'Your household income is $30,000-$50,000.',
        '$50,000-$75,000': 'Your household income is $50,000-$75,000.',
        '$75,000-$100,000': 'Your household income is $75,000-$100,000.',
        '$100,000 or more': 'Your household income is $100,000 or more.'
    },
    'POLIDEOLOGY': {
        'Very conservative': 'Your political ideology is very conservative.',
        'Conservative': 'Your political ideology is conservative.',
        'Moderate': 'Your political ideology is moderate.',
        'Liberal': 'Your political ideology is liberal.',
        'Very liberal': 'Your political ideology is very liberal.'
    },
    'RACE': {
        'White': 'Your race/ethnicity is White.',
        'Black': 'Your race/ethnicity is Black.',
        'Asian': 'Your race/ethnicity is Asian.',
        'Hispanic': 'Your race/ethnicity is Hispanic.',
        'Mixed Race': 'Your race/ethnicity is Mixed Race.',
        'Other': 'Your race/ethnicity is other.'
    }
}

# Values to skip (empty or refused)
SKIP_VALUES = {'', 'Refused'}


def print_dataset_statistics(data_list: List[Dict], output: List[Dict]) -> Dict:
    """
    Print and return dataset statistics after preprocessing.
    
    Args:
        data_list: Original list of survey instances
        output: Processed output list
    
    Returns:
        Dictionary containing all statistics
    """
    # 1. Number of unique respondents
    unique_respondent_ids = set()
    for instance in data_list:
        if 'id' in instance:
            unique_respondent_ids.add(instance['id'])
    num_unique_respondents = len(unique_respondent_ids) if unique_respondent_ids else len(data_list)
    
    # 2. Total survey responses and average per respondent
    total_responses = 0
    for instance in data_list:
        if 'responses' in instance:
            total_responses += len(instance['responses'])
    
    avg_responses_per_respondent = total_responses / num_unique_respondents if num_unique_respondents > 0 else 0
    
    # 3. Unique number of questions
    unique_questions = set()
    for instance in data_list:
        if 'responses' in instance:
            for response in instance['responses']:
                if 'Question_var' in response:
                    unique_questions.add(response['Question_var'])
    num_unique_questions = len(unique_questions)
    
    # 4. Unique number of demographic groups (from processed output)
    unique_demographic_groups = set()
    for item in output:
        if item['num_grouping_vars'] == 1:
            # Extract the demographic key-value pair
            for demo_key, demo_value in item['group_prompt_variable_map'].items():
                unique_demographic_groups.add((demo_key, demo_value))
    num_unique_demographic_groups = len(unique_demographic_groups)
    
    # 5. Total number of instances
    total_instances = len(output)
    
    # Count instances by type
    overall_instances = sum(1 for item in output if item['num_grouping_vars'] == 0)
    demographic_instances = sum(1 for item in output if item['num_grouping_vars'] == 1)
    
    # Print statistics
    print("\n" + "="*60)
    print("DATASET STATISTICS")
    print("="*60)
    print(f"Number of unique respondents: {num_unique_respondents}")
    print(f"AVG survey response per respondent: {avg_responses_per_respondent:.2f}")
    print(f"Unique number of questions: {num_unique_questions}")
    print(f"Unique number of demographic groups: {num_unique_demographic_groups}")
    print(f"Total number of instances: {total_instances}")
    print("-"*60)
    print(f"  - Overall instances (num_grouping_vars=0): {overall_instances}")
    print(f"  - Demographic instances (num_grouping_vars=1): {demographic_instances}")
    print("="*60 + "\n")
    
    # Return statistics as dictionary
    return {
        'num_unique_respondents': num_unique_respondents,
        'avg_responses_per_respondent': avg_responses_per_respondent,
        'num_unique_questions': num_unique_questions,
        'num_unique_demographic_groups': num_unique_demographic_groups,
        'total_instances': total_instances,
        'overall_instances': overall_instances,
        'demographic_instances': demographic_instances
    }


def preprocess_survey_data(
    data_list: List[Dict], 
    wave: float, 
    year: int = 2017,
    min_group_size: int = 0,
    min_group_size_overall: Optional[int] = None,
    print_stats: bool = True
) -> List[Dict]:
    """
    Preprocess survey data to create grouped responses.
    
    Args:
        data_list: List of survey instances, each with 'id', 'feature', and 'responses'
        wave: Wave number for the survey
        year: Survey year
        min_group_size: Minimum group size threshold for demographic groups (num_grouping_vars=1).
                        Groups with size below this threshold will be filtered out.
        min_group_size_overall: Minimum group size threshold for overall groups (num_grouping_vars=0).
                                If None, no filtering is applied to overall groups.
        print_stats: Whether to print dataset statistics after preprocessing
    
    Returns:
        List of processed instances with grouped distributions
    """
    
    # Step 1: Extract all unique questions and their metadata
    questions = {}
    for instance in data_list:
        for response in instance['responses']:
            q_var = response['Question_var']
            if q_var not in questions:
                # Parse possible answers (handle string representation of list)
                possible_answers = response['Possible_Answers']
                if isinstance(possible_answers, str):
                    possible_answers = eval(possible_answers)
                
                questions[q_var] = {
                    'question': response['Question'],
                    'possible_answers': possible_answers,
                    'question_var': q_var
                }
    
    # Step 2: Get all demographic feature keys
    demographic_keys = list(DEMOGRAPHIC_VALUES.keys())
    
    # Step 3: Collect all responses grouped by question variable
    question_responses = defaultdict(list)  # q_var -> list of {'response': str, 'features': dict}
    
    for instance in data_list:
        features = instance['feature']
        for response in instance['responses']:
            q_var = response['Question_var']
            resp_value = response['Response']
            question_responses[q_var].append({
                'response': resp_value,
                'features': features
            })
    
    # Step 4: Helper function to create answer options mapping
    def create_answer_options(possible_answers: List[str]) -> Dict[str, str]:
        answer_options = {}
        letters = string.ascii_uppercase
        for i, ans in enumerate(possible_answers):
            if i < len(letters):
                answer_options[letters[i]] = ans
        return answer_options
    
    # Step 5: Helper function to create input template
    def create_input_template(question: str, answer_options: Dict[str, str]) -> str:
        input_template = f"{question}\n\nOptions:\n"
        for letter in sorted(answer_options.keys()):
            input_template += f"({letter}): {answer_options[letter]}\n"
        return input_template.strip()
    
    # Step 6: Helper function to calculate answer distribution
    def calculate_distribution(responses: List[Dict], answer_options: Dict[str, str]) -> tuple:
        answer_counts = defaultdict(int)
        valid_responses = [r for r in responses if r['response']]  # Only count non-empty responses
        total = len(valid_responses)
        
        for r in valid_responses:
            answer_counts[r['response']] += 1
        
        human_answer = {}
        for letter, ans in answer_options.items():
            count = answer_counts.get(ans, 0)
            human_answer[letter] = (count / total * 100) if total > 0 else 0.0
        
        return human_answer, total
    
    # Step 7: Helper function to get demographic description
    def get_demographic_description(demo_key: str, demo_value: str) -> str:
        if demo_key in DEMOGRAPHIC_DESCRIPTIONS:
            if demo_value in DEMOGRAPHIC_DESCRIPTIONS[demo_key]:
                return DEMOGRAPHIC_DESCRIPTIONS[demo_key][demo_value]
        # Fallback for values not in descriptions
        return f'Your {demo_key} is {demo_value}.'
    
    # Step 8: Create output list
    output = []
    
    # Track filtered groups for reporting
    filtered_overall = 0
    filtered_demographic = 0
    
    # Get ordered list of question variables
    question_vars = list(questions.keys())
    
    # ==========================================
    # FIRST N INSTANCES: Overall distributions (num_grouping_vars = 0)
    # ==========================================
    for question_idx, q_var in enumerate(question_vars):
        q_info = questions[q_var]
        responses_list = question_responses[q_var]
        
        # Create answer options
        answer_options = create_answer_options(q_info['possible_answers'])
        
        # Create input template
        input_template = create_input_template(q_info['question'], answer_options)
        
        # Calculate overall distribution
        human_answer, group_size = calculate_distribution(responses_list, answer_options)
        
        # Skip questions with no valid responses
        if group_size == 0:
            continue
        
        # Apply threshold for overall groups if specified
        if min_group_size_overall is not None and group_size < min_group_size_overall:
            filtered_overall += 1
            continue
        
        output.append({
            'group_prompt_template': f'You are from the United States. The year is {year}.',
            'group_prompt_variable_map': {},
            'input_template': input_template,
            'input_variable_map': {},
            'output_format': 'choice',
            'answer_options': answer_options,
            'auxiliary': {},
            'group_size': group_size,
            'human_answer': human_answer,
            'wave': wave,
            'num_grouping_vars': 0,
            'dataset_name': 'OpinionQA',
            'question_idx': question_idx
        })
    
    # ==========================================
    # AFTER N: Grouped by demographic features (num_grouping_vars = 1)
    # ==========================================
    for question_idx, q_var in enumerate(question_vars):
        q_info = questions[q_var]
        responses_list = question_responses[q_var]
        
        # Create answer options
        answer_options = create_answer_options(q_info['possible_answers'])
        
        # Create input template
        input_template = create_input_template(q_info['question'], answer_options)
        
        # Group by each demographic feature
        for demo_key in demographic_keys:
            # Get unique values for this demographic from the data
            demo_values = set(r['features'].get(demo_key) for r in responses_list)
            
            for demo_value in sorted(demo_values):
                # Skip empty values and 'Refused'
                if demo_value in SKIP_VALUES:
                    continue
                
                # Filter responses for this demographic value
                filtered_responses = [
                    r for r in responses_list 
                    if r['features'].get(demo_key) == demo_value
                ]
                
                if not filtered_responses:
                    continue
                
                # Calculate distribution for this demographic group
                human_answer, group_size = calculate_distribution(filtered_responses, answer_options)
                
                # Skip groups with no valid responses
                if group_size == 0:
                    continue
                
                # Apply minimum group size threshold
                if group_size < min_group_size:
                    filtered_demographic += 1
                    continue
                
                # Get demographic description
                demo_description = get_demographic_description(demo_key, demo_value)
                
                # Create group prompt
                group_prompt = f'You are from the United States. The year is {year}. {demo_description}'
                
                output.append({
                    'group_prompt_template': group_prompt,
                    'group_prompt_variable_map': {demo_key: demo_value},
                    'input_template': input_template,
                    'input_variable_map': {},
                    'output_format': 'choice',
                    'answer_options': answer_options,
                    'auxiliary': {},
                    'group_size': group_size,
                    'human_answer': human_answer,
                    'wave': wave,
                    'num_grouping_vars': 1,
                    'dataset_name': 'OpinionQA',
                    'question_idx': question_idx
                })
    
    # Print filtering summary
    print(f"\nFiltering Summary:")
    print(f"  - Minimum group size for demographic groups: {min_group_size}")
    print(f"  - Minimum group size for overall groups: {min_group_size_overall if min_group_size_overall is not None else 'No threshold'}")
    print(f"  - Filtered out {filtered_overall} overall instances")
    print(f"  - Filtered out {filtered_demographic} demographic group instances")
    
    # Print dataset statistics
    if print_stats:
        print_dataset_statistics(data_list, output)
    
    return output


def load_and_preprocess(
    input_file: str, 
    output_file: str, 
    wave: float, 
    year: int = 2017,
    min_group_size: int = 0,
    min_group_size_overall: Optional[int] = None,
    print_stats: bool = True
):
    """
    Load data from file, preprocess, and save results.
    
    Args:
        input_file: Path to input JSON file containing list of survey instances
        output_file: Path to output JSON file
        wave: Wave number for the survey
        year: Survey year
        min_group_size: Minimum group size threshold for demographic groups
        min_group_size_overall: Minimum group size threshold for overall groups
        print_stats: Whether to print dataset statistics
    """
    # Load data
    with open(input_file, 'r') as f:
        data_list = json.load(f)
    
    # Preprocess
    processed_data = preprocess_survey_data(
        data_list, 
        wave=wave, 
        year=year,
        min_group_size=min_group_size,
        min_group_size_overall=min_group_size_overall,
        print_stats=print_stats
    )
    
    # Save results
    with open(output_file, 'w') as f:
        json.dump(processed_data, f, indent=2)
    
    print(f"Output saved to {output_file}")
    
    return processed_data


def get_group_size_statistics(data_list: List[Dict], wave: float, year: int = 2017) -> Dict:
    """
    Get statistics about group sizes to help determine appropriate threshold.
    
    Args:
        data_list: List of survey instances
        wave: Wave number
        year: Survey year
    
    Returns:
        Dictionary with group size statistics
    """
    # Process without filtering to get all group sizes
    processed = preprocess_survey_data(data_list, wave=wave, year=year, min_group_size=0, print_stats=False)
    
    overall_sizes = []
    demographic_sizes = []
    demographic_sizes_by_key = defaultdict(list)
    
    for item in processed:
        if item['num_grouping_vars'] == 0:
            overall_sizes.append(item['group_size'])
        else:
            demographic_sizes.append(item['group_size'])
            demo_key = list(item['group_prompt_variable_map'].keys())[0]
            demographic_sizes_by_key[demo_key].append(item['group_size'])
    
    def calc_stats(sizes):
        if not sizes:
            return {}
        sizes = sorted(sizes)
        n = len(sizes)
        return {
            'count': n,
            'min': sizes[0],
            'max': sizes[-1],
            'mean': sum(sizes) / n,
            'median': sizes[n // 2],
            'p10': sizes[int(n * 0.1)] if n >= 10 else sizes[0],
            'p25': sizes[int(n * 0.25)] if n >= 4 else sizes[0],
            'p75': sizes[int(n * 0.75)] if n >= 4 else sizes[-1],
            'p90': sizes[int(n * 0.9)] if n >= 10 else sizes[-1]
        }
    
    stats = {
        'overall': calc_stats(overall_sizes),
        'demographic_all': calc_stats(demographic_sizes),
        'demographic_by_key': {k: calc_stats(v) for k, v in demographic_sizes_by_key.items()}
    }
    
    return stats


def validate_demographics(data_list: List[Dict]) -> Dict[str, List[str]]:
    """Check for any demographic values not in the predefined list."""
    invalid_values = defaultdict(list)
    for instance in data_list:
        for key, value in instance['feature'].items():
            if key in DEMOGRAPHIC_VALUES:
                if value not in DEMOGRAPHIC_VALUES[key]:
                    if value not in invalid_values[key]:
                        invalid_values[key].append(value)
    return dict(invalid_values)


def get_demographic_summary(data_list: List[Dict]) -> Dict[str, Dict[str, int]]:
    """Get count of each demographic value in the data."""
    demo_counts = defaultdict(lambda: defaultdict(int))
    for instance in data_list:
        for key, value in instance['feature'].items():
            demo_counts[key][value] += 1
    return {k: dict(v) for k, v in demo_counts.items()}


In [ ]:
processed = preprocess_survey_data(data, wave=wave, year=wave_year_map[int(wave)], min_group_size=100)


In [ ]:
group=[]
cnt=0
for d in processed:
    if d['group_prompt_variable_map']:
        group.append(str(d['group_prompt_variable_map']))
    else:
        cnt+=1
print(len(set(group)))
print(cnt)

In [ ]:
extended=[{'RACE': 'Mixed Race'},
    {'RACE': 'Other'},
    {'SEX': 'In some other way'},
    {'MARITAL': 'Separated'},
    {'RELIG': 'Mormon'},
    {'RELIG': 'Jewish'},
    {'RELIG': 'Hindu'},
    {'RELIG': 'Buddhist'},
    {'RELIG': 'Muslim'},
    {'RELIG': 'Orthodox'},
    {'RELIG': 'Christian'},
    {'RELIG': 'Unitarian'},
    {'RELIG': 'Other'}]
for d in processed:
    if d['group_prompt_variable_map'] in extended:
        print(d)

In [ ]:
with open(f'./dataset/opinionqa/American_Trends_Panel_W{wave}/opinionqa_W{wave}_grouped.json', 'w') as f:
    json.dump(processed, f)

In [ ]:
with open(f'./dataset/opinionqa/American_Trends_Panel_W{wave}/opinionqa_W{wave}_grouped.json') as f:
    check=json.load(f)

In [ ]:
check[0]

In [ ]:
check[-2]

In [ ]:
check[100]['human_answer'].keys()

In [ ]:
var=[]
for d in check:
    if len(d['group_prompt_variable_map']):
        var.append(list(d['group_prompt_variable_map'].keys())[0])
print(set(var))

### GSS

In [ ]:
data_dir='./dataset/gss/gss_per_user_with_gt.json' 


In [ ]:
with open(data_dir) as f:
    gss=json.load(f)

In [ ]:
gss_f=defaultdict(list)
for d in gss:
    for k,v in d['feature'].items():
        gss_f[k].append(v)

In [ ]:
for k,v in gss_f.items():
    gss_f[k]=list(set(gss_f[k]))

In [ ]:
gss_f

In [ ]:
def map_instance_features(instance):
    """Map individual instance features to reference format, keeping unmapped keys."""
    original_features = instance['feature']
    responses = {r['Question_var']: r['Response'] for r in instance['responses']}
    
    # Start with original features (keep unmapped keys)
    mapped_features = dict(original_features)
    
    # AGE mapping (numeric to range)
    age_str = original_features.get('age', '')
    if age_str:
        if age_str == '89 or older':
            mapped_features['AGE'] = '65+'
        else:
            try:
                age = float(age_str)
                if 18 <= age <= 29:
                    mapped_features['AGE'] = '18-29'
                elif 30 <= age <= 49:
                    mapped_features['AGE'] = '30-49'
                elif 50 <= age <= 64:
                    mapped_features['AGE'] = '50-64'
                elif age >= 65:
                    mapped_features['AGE'] = '65+'
                else:
                    mapped_features['AGE'] = ''
            except:
                mapped_features['AGE'] = ''
    else:
        mapped_features['AGE'] = ''
    # Remove original key after mapping
    if 'age' in mapped_features:
        del mapped_features['age']
    
    # CITIZEN mapping (from 'born')
    born = original_features.get('born', '')
    citizen_map = {'yes': 'Yes', 'no': 'No'}
    mapped_features['CITIZEN'] = citizen_map.get(born, '')
    if 'born' in mapped_features:
        del mapped_features['born']
    
    # EDUCATION mapping (from 'degree')
    degree = original_features.get('degree', '')
    education_map = {
        "bachelor's": "College graduate/some postgrad",
        "high school": "High school graduate",
        "associate/junior college": "Associate's degree",
        "less than high school": "Less than high school",
        "graduate": "Postgraduate"
    }
    mapped_features['EDUCATION'] = education_map.get(degree, '')
    if 'degree' in mapped_features:
        del mapped_features['degree']
    
    # INCOME mapping
    income = original_features.get('income', '')
    if income == '':
        mapped_features['INCOME'] = ''
    elif income == '$25,000 or more':
        mapped_features['INCOME'] = '$30,000-$50,000'
    elif income in ['$20,000 to $24,999', '$15,000 to $19,999', '$10,000 to $14,999',
                    '$8,000 to $9,999', '$7,000 to $7,999', '$6,000 to $6,999',
                    '$5,000 to $5,999', '$4,000 to $4,999', '$3,000 to $3,999',
                    '$1,000 to $2,999', 'under $1,000']:
        mapped_features['INCOME'] = 'Less than $30,000'
    else:
        mapped_features['INCOME'] = ''
    if 'income' in mapped_features:
        del mapped_features['income']
    
    # MARITAL mapping
    marital = original_features.get('marital', '')
    marital_map = {
        'married': 'Married',
        'separated': 'Separated',
        'widowed': 'Widowed',
        'divorced': 'Divorced',
        'never married': 'Never been married'
    }
    mapped_features['MARITAL'] = marital_map.get(marital, '')
    if 'marital' in mapped_features:
        del mapped_features['marital']
    
    # RACE mapping (combining race and hispanic)
    race = original_features.get('race', '')
    hispanic = original_features.get('hispanic', '')
    
    if hispanic and hispanic != 'not hispanic' and hispanic != '':
        mapped_features['RACE'] = 'Hispanic'
    else:
        race_map = {'white': 'White', 'black': 'Black', 'other': 'Other'}
        mapped_features['RACE'] = race_map.get(race, '')
    if 'race' in mapped_features:
        del mapped_features['race']
    if 'hispanic' in mapped_features:
        del mapped_features['hispanic']
    
    # CREGION mapping (from 'region')
    region = original_features.get('region', '')
    region_map = {'south': 'South', 'west': 'West', 'midwest': 'Midwest', 'northeast': 'Northeast'}
    mapped_features['CREGION'] = region_map.get(region, '')
    if 'region' in mapped_features:
        del mapped_features['region']
    
    # RELIG mapping
    relig = original_features.get('relig', '')
    relig_map = {
        'protestant': 'Protestant',
        'catholic': 'Roman Catholic',
        'jewish': 'Jewish',
        'muslim/islam': 'Muslim',
        'buddhism': 'Buddhist',
        'hinduism': 'Hindu',
        'christian': 'Christian',
        'orthodox-christian': 'Orthodox',
        'none': 'Nothing in particular',
        'other': 'Other',
        'inter-nondenominational': 'Other',
        'other eastern religions': 'Other',
        'native american': 'Other'
    }
    mapped_features['RELIG'] = relig_map.get(relig, '')
    if 'relig' in mapped_features:
        del mapped_features['relig']
    
    # SEX mapping
    sex = original_features.get('sex', '')
    sex_map = {'male': 'Male', 'female': 'Female'}
    mapped_features['SEX'] = sex_map.get(sex, '')
    if 'sex' in mapped_features:
        del mapped_features['sex']
    
    # RELIGATTEND mapping (from 'attend' response)
    attend = responses.get('attend', '')
    attend_map = {
        'never': 'Never',
        'less than once a year': 'Seldom',
        'once a year': 'A few times a year',
        'several times a year': 'A few times a year',
        'once a month': 'Once or twice a month',
        '2-3 times a month': 'Once or twice a month',
        'nearly every week': 'Once a week',
        'every week': 'Once a week',
        'more than once a week': 'More than once a week'
    }
    mapped_features['RELIGATTEND'] = attend_map.get(attend, '')
    
    # POLPARTY mapping (from 'partyid' response)
    partyid = responses.get('partyid', '')
    if partyid:
        if 'democrat' in partyid.lower():
            mapped_features['POLPARTY'] = 'Democrat'
        elif 'republican' in partyid.lower():
            mapped_features['POLPARTY'] = 'Republican'
        elif 'independent' in partyid.lower():
            mapped_features['POLPARTY'] = 'Independent'
        else:
            mapped_features['POLPARTY'] = ''
    else:
        mapped_features['POLPARTY'] = ''
    
    # POLIDEOLOGY mapping (from 'polviews' response)
    polviews = responses.get('polviews', '')
    if polviews:
        polviews_lower = polviews.lower()
        if 'extremely liberal' in polviews_lower:
            mapped_features['POLIDEOLOGY'] = 'Very liberal'
        elif 'extremely conservative' in polviews_lower:
            mapped_features['POLIDEOLOGY'] = 'Very conservative'
        elif 'slightly liberal' in polviews_lower or polviews_lower == 'liberal':
            mapped_features['POLIDEOLOGY'] = 'Liberal'
        elif 'slightly conservative' in polviews_lower or polviews_lower == 'conservative':
            mapped_features['POLIDEOLOGY'] = 'Conservative'
        elif 'moderate' in polviews_lower:
            mapped_features['POLIDEOLOGY'] = 'Moderate'
        else:
            mapped_features['POLIDEOLOGY'] = ''
    else:
        mapped_features['POLIDEOLOGY'] = ''
    
    return {
        'id': instance['id'],
        'feature': mapped_features,
        'responses': instance['responses']
    }


def process_gss(gss):
    """
    Process list of GSS instances and map features to reference format.
    Keeps unmapped keys like 'indus10', 'occ10', 'childs', 'educ'.
    
    Args:
        gss: List of instance dictionaries with 'id', 'feature', and 'responses'
        
    Returns:
        processed: List of processed instances with mapped features
    """
    processed = []
    
    for instance in gss:
        try:
            mapped_instance = map_instance_features(instance)
            processed.append(mapped_instance)
        except Exception as e:
            print(f"Error processing instance {instance.get('id', 'unknown')}: {e}")
            continue
    
    return processed



In [ ]:
processed_gss = process_gss(gss)


In [ ]:
with open('./dataset/gss/gss_per_user_with_gt_opinionqa_format.json', 'w') as f:
    json.dump(processed_gss, f)

### GSS preprocess grouped

In [ ]:
import json
from collections import defaultdict
import string
from typing import List, Dict, Any, Optional

# Define all possible demographic values (only these will be used for grouping)
DEMOGRAPHIC_VALUES = {
    'CREGION': ['Midwest', 'Northeast', 'South', 'West'],
    'AGE': ['', '18-29', '65+', 'Refused', '50-64', '30-49'],
    'SEX': ['In some other way', 'Female', 'Male', 'Refused'],
    'EDUCATION': ['Some college, no degree',
                  'Refused',
                  "Associate's degree",
                  'College graduate/some postgrad',
                  'Less than high school',
                  'Postgraduate',
                  'High school graduate'],
    'MARITAL': ['Living with a partner',
                '',
                'Refused',
                'Married',
                'Widowed',
                'Divorced',
                'Separated',
                'Never been married'],
    'RELIG': ['',
              'Mormon',
              'Atheist',
              'Refused',
              'Jewish',
              'Hindu',
              'Roman Catholic',
              'Unitarian',
              'Christian',
              'Nothing in particular',
              'Other',
              'Buddhist',
              'Agnostic',
              'Protestant',
              'Muslim',
              'Orthodox'],
    'RELIGATTEND': ['',
                    'A few times a year',
                    'Never',
                    'Refused',
                    'Once or twice a month',
                    'Once a week',
                    'More than once a week',
                    'Seldom'],
    'POLPARTY': ['',
                 'Republican',
                 'Refused',
                 'Other',
                 'Democrat',
                 'Independent'],
    'INCOME': ['',
               'Refused',
               '$100,000 or more',
               '$75,000-$100,000',
               'Less than $30,000',
               '$50,000-$75,000',
               '$30,000-$50,000'],
    'POLIDEOLOGY': ['',
                    'Conservative',
                    'Very liberal',
                    'Refused',
                    'Moderate',
                    'Liberal',
                    'Very conservative'],
    'RACE': ['Hispanic',
             'Refused',
             'Mixed Race',
             'White',
             'Black',
             'Other',
             'Asian']
}

# Define demographic description templates
DEMOGRAPHIC_DESCRIPTIONS = {
    'CREGION': {
        'Midwest': 'You are from the Midwest region.',
        'Northeast': 'You are from the Northeast region.',
        'South': 'You are from the South region.',
        'West': 'You are from the West region.'
    },
    'AGE': {
        '18-29': 'You are aged 18-29.',
        '30-49': 'You are aged 30-49.',
        '50-64': 'You are aged 50-64.',
        '65+': 'You are aged 65 or older.'
    },
    'SEX': {
        'Female': 'You are Female.',
        'Male': 'You are Male.',
        'In some other way': 'You identify your gender in some other way.'
    },
    'EDUCATION': {
        'Less than high school': 'Your highest education level is less than high school.',
        'High school graduate': 'Your highest education level is high school graduate.',
        'Some college, no degree': 'Your highest education level is some college, no degree.',
        "Associate's degree": "Your highest education level is an Associate's degree.",
        'College graduate/some postgrad': 'Your highest education level is college graduate or some postgraduate.',
        'Postgraduate': 'Your highest education level is postgraduate.'
    },
    'MARITAL': {
        'Married': 'You are married.',
        'Living with a partner': 'You are living with a partner.',
        'Widowed': 'You are widowed.',
        'Divorced': 'You are divorced.',
        'Separated': 'You are separated.',
        'Never been married': 'You have never been married.'
    },
    'RELIG': {
        'Protestant': 'Your religion is Protestant.',
        'Roman Catholic': 'Your religion is Roman Catholic.',
        'Mormon': 'Your religion is Mormon.',
        'Orthodox': 'Your religion is Orthodox Christian.',
        'Jewish': 'Your religion is Jewish.',
        'Muslim': 'Your religion is Muslim.',
        'Buddhist': 'Your religion is Buddhist.',
        'Hindu': 'Your religion is Hindu.',
        'Atheist': 'You are an Atheist.',
        'Agnostic': 'You are Agnostic.',
        'Nothing in particular': 'You have no particular religion.',
        'Unitarian': 'Your religion is Unitarian.',
        'Christian': 'Your religion is Christian.',
        'Other': 'Your religion is other.'
    },
    'RELIGATTEND': {
        'More than once a week': 'You attend religious services more than once a week.',
        'Once a week': 'You attend religious services once a week.',
        'Once or twice a month': 'You attend religious services once or twice a month.',
        'A few times a year': 'You attend religious services a few times a year.',
        'Seldom': 'You seldom attend religious services.',
        'Never': 'You never attend religious services.'
    },
    'POLPARTY': {
        'Republican': 'You identify as a Republican.',
        'Democrat': 'You identify as a Democrat.',
        'Independent': 'You identify as an Independent.',
        'Other': 'You identify with another political party.'
    },
    'INCOME': {
        'Less than $30,000': 'Your household income is less than $30,000.',
        '$30,000-$50,000': 'Your household income is $30,000-$50,000.',
        '$50,000-$75,000': 'Your household income is $50,000-$75,000.',
        '$75,000-$100,000': 'Your household income is $75,000-$100,000.',
        '$100,000 or more': 'Your household income is $100,000 or more.'
    },
    'POLIDEOLOGY': {
        'Very conservative': 'Your political ideology is very conservative.',
        'Conservative': 'Your political ideology is conservative.',
        'Moderate': 'Your political ideology is moderate.',
        'Liberal': 'Your political ideology is liberal.',
        'Very liberal': 'Your political ideology is very liberal.'
    },
    'RACE': {
        'White': 'Your race/ethnicity is White.',
        'Black': 'Your race/ethnicity is Black.',
        'Asian': 'Your race/ethnicity is Asian.',
        'Hispanic': 'Your race/ethnicity is Hispanic.',
        'Mixed Race': 'Your race/ethnicity is Mixed Race.',
        'Other': 'Your race/ethnicity is other.'
    }
}

# Values to skip (empty or refused)
SKIP_VALUES = {'', 'Refused'}

# Features to use for demographic grouping (ignore others like 'childs', 'educ', 'indus10', 'occ10')
DEMOGRAPHIC_KEYS = ['CREGION', 'AGE', 'SEX', 'EDUCATION','MARITAL', 
                    'RELIG', 'RELIGATTEND', 'POLPARTY', 'INCOME', 'POLIDEOLOGY', 'RACE']


def parse_possible_answers(possible_answers_str: str) -> List[str]:
    """
    Parse possible answers from string format.
    Handles both semicolon-separated strings and list representations.
    
    Args:
        possible_answers_str: String containing possible answers
        
    Returns:
        List of possible answers
    """
    if isinstance(possible_answers_str, list):
        return possible_answers_str
    
    # Check if it's a string representation of a list
    if possible_answers_str.strip().startswith('['):
        return eval(possible_answers_str)
    
    # Semicolon-separated format
    return [ans.strip() for ans in possible_answers_str.split(';')]


def normalize_response(response: str, possible_answers: List[str]) -> str:
    """
    Normalize a response to match one of the possible answers (case-insensitive).
    
    Args:
        response: The response string
        possible_answers: List of possible answers
        
    Returns:
        Matched possible answer or empty string if no match
    """
    if not response:
        return ''
    
    response_lower = response.lower().strip()
    
    # Create a mapping of lowercase answers to original answers
    answer_map = {ans.lower().strip(): ans for ans in possible_answers}
    
    # Direct match
    if response_lower in answer_map:
        return answer_map[response_lower]
    
    # Partial match for common variations
    for ans_lower, ans_original in answer_map.items():
        # Check if response is contained in answer or vice versa
        if response_lower in ans_lower or ans_lower in response_lower:
            return ans_original
    
    # No match found - return original response (will be counted if it matches exactly)
    return response


def preprocess_survey_data(
    data_list: List[Dict], 
    wave: float, 
    year: int = 2017,
    min_group_size: int = 0,
    min_group_size_overall: Optional[int] = None,
    dataset_name: str = 'OpinionQA'
) -> List[Dict]:
    """
    Preprocess survey data to create grouped responses.
    
    Args:
        data_list: List of survey instances, each with 'id', 'feature', and 'responses'
        wave: Wave number for the survey
        year: Survey year
        min_group_size: Minimum group size threshold for demographic groups (num_grouping_vars=1).
                        Groups with size below this threshold will be filtered out.
        min_group_size_overall: Minimum group size threshold for overall groups (num_grouping_vars=0).
                                If None, no filtering is applied to overall groups.
        dataset_name: Name of the dataset
    
    Returns:
        List of processed instances with grouped distributions
    """
    
    # Step 1: Extract all unique questions and their metadata
    questions = {}
    for instance in data_list:
        for response in instance['responses']:
            q_var = response['Question_var']
            if q_var not in questions:
                # Parse possible answers
                possible_answers = parse_possible_answers(response['Possible_Answers'])
                
                questions[q_var] = {
                    'question': response['Question'],
                    'possible_answers': possible_answers,
                    'question_var': q_var
                }
    
    # Step 2: Collect all responses grouped by question variable
    question_responses = defaultdict(list)  # q_var -> list of {'response': str, 'features': dict}
    
    for instance in data_list:
        # Extract only demographic features
        features = {k: v for k, v in instance['feature'].items() if k in DEMOGRAPHIC_KEYS}
        
        for response in instance['responses']:
            q_var = response['Question_var']
            resp_value = response['Response']
            
            # Normalize the response to match possible answers
            possible_answers = questions[q_var]['possible_answers']
            normalized_response = normalize_response(resp_value, possible_answers)
            
            question_responses[q_var].append({
                'response': normalized_response,
                'features': features
            })
    
    # Step 3: Helper function to create answer options mapping
    def create_answer_options(possible_answers: List[str]) -> Dict[str, str]:
        answer_options = {}
        letters = string.ascii_uppercase
        for i, ans in enumerate(possible_answers):
            if i < len(letters):
                answer_options[letters[i]] = ans
        return answer_options
    
    # Step 4: Helper function to create input template
    def create_input_template(question: str, answer_options: Dict[str, str]) -> str:
        input_template = f"{question}\n\nOptions:\n"
        for letter in sorted(answer_options.keys()):
            input_template += f"({letter}): {answer_options[letter]}\n"
        return input_template.strip()
    
    # Step 5: Helper function to calculate answer distribution
    def calculate_distribution(responses: List[Dict], answer_options: Dict[str, str]) -> tuple:
        answer_counts = defaultdict(int)
        valid_responses = [r for r in responses if r['response']]  # Only count non-empty responses
        total = len(valid_responses)
        
        for r in valid_responses:
            answer_counts[r['response']] += 1
        
        human_answer = {}
        for letter, ans in answer_options.items():
            count = answer_counts.get(ans, 0)
            human_answer[letter] = (count / total * 100) if total > 0 else 0.0
        
        return human_answer, total
    
    # Step 6: Helper function to get demographic description
    def get_demographic_description(demo_key: str, demo_value: str) -> str:
        if demo_key in DEMOGRAPHIC_DESCRIPTIONS:
            if demo_value in DEMOGRAPHIC_DESCRIPTIONS[demo_key]:
                return DEMOGRAPHIC_DESCRIPTIONS[demo_key][demo_value]
        # Fallback for values not in descriptions
        return f'Your {demo_key} is {demo_value}.'
    
    # Step 7: Create output list
    output = []
    
    # Track filtered groups for reporting
    filtered_overall = 0
    filtered_demographic = 0
    
    # Get ordered list of question variables
    question_vars = list(questions.keys())
    
    # ==========================================
    # FIRST N INSTANCES: Overall distributions (num_grouping_vars = 0)
    # ==========================================
    for question_idx, q_var in enumerate(question_vars):
        q_info = questions[q_var]
        responses_list = question_responses[q_var]
        
        # Create answer options
        answer_options = create_answer_options(q_info['possible_answers'])
        
        # Create input template
        input_template = create_input_template(q_info['question'], answer_options)
        
        # Calculate overall distribution
        human_answer, group_size = calculate_distribution(responses_list, answer_options)
        
        # Skip questions with no valid responses
        if group_size == 0:
            continue
        
        # Apply threshold for overall groups if specified
        if min_group_size_overall is not None and group_size < min_group_size_overall:
            filtered_overall += 1
            continue
        
        output.append({
            'group_prompt_template': f'You are from the United States. The year is {year}.',
            'group_prompt_variable_map': {},
            'input_template': input_template,
            'input_variable_map': {},
            'output_format': 'choice',
            'answer_options': answer_options,
            'auxiliary': {},
            'group_size': group_size,
            'human_answer': human_answer,
            'wave': wave,
            'num_grouping_vars': 0,
            'dataset_name': dataset_name,
            'question_idx': question_idx
        })
    
    # ==========================================
    # AFTER N: Grouped by demographic features (num_grouping_vars = 1)
    # ==========================================
    for question_idx, q_var in enumerate(question_vars):
        q_info = questions[q_var]
        responses_list = question_responses[q_var]
        
        # Create answer options
        answer_options = create_answer_options(q_info['possible_answers'])
        
        # Create input template
        input_template = create_input_template(q_info['question'], answer_options)
        
        # Group by each demographic feature
        for demo_key in DEMOGRAPHIC_KEYS:
            # Get unique values for this demographic from the data
            demo_values = set(r['features'].get(demo_key) for r in responses_list if r['features'].get(demo_key) is not None)
            
            for demo_value in sorted(demo_values):
                # Skip empty values and 'Refused'
                if demo_value in SKIP_VALUES:
                    continue
                
                # Filter responses for this demographic value
                filtered_responses = [
                    r for r in responses_list 
                    if r['features'].get(demo_key) == demo_value
                ]
                
                if not filtered_responses:
                    continue
                
                # Calculate distribution for this demographic group
                human_answer, group_size = calculate_distribution(filtered_responses, answer_options)
                
                # Skip groups with no valid responses
                if group_size == 0:
                    continue
                
                # Apply minimum group size threshold
                if group_size < min_group_size:
                    filtered_demographic += 1
                    continue
                
                # Get demographic description
                demo_description = get_demographic_description(demo_key, demo_value)
                
                # Create group prompt
                group_prompt = f'You are from the United States. The year is {year}. {demo_description}'
                
                output.append({
                    'group_prompt_template': group_prompt,
                    'group_prompt_variable_map': {demo_key: demo_value},
                    'input_template': input_template,
                    'input_variable_map': {},
                    'output_format': 'choice',
                    'answer_options': answer_options,
                    'auxiliary': {},
                    'group_size': group_size,
                    'human_answer': human_answer,
                    'wave': wave,
                    'num_grouping_vars': 1,
                    'dataset_name': dataset_name,
                    'question_idx': question_idx
                })
    
    # Print filtering summary
    print(f"Filtering Summary:")
    print(f"  - Total unique questions: {len(question_vars)}")
    print(f"  - Minimum group size for demographic groups: {min_group_size}")
    print(f"  - Minimum group size for overall groups: {min_group_size_overall if min_group_size_overall is not None else 'No threshold'}")
    print(f"  - Filtered out {filtered_overall} overall instances")
    print(f"  - Filtered out {filtered_demographic} demographic group instances")
    print(f"  - Total instances in output: {len(output)}")
    
    return output


def load_and_preprocess(
    input_file: str, 
    output_file: str, 
    wave: float, 
    year: int = 2017,
    min_group_size: int = 0,
    min_group_size_overall: Optional[int] = None,
    dataset_name: str = 'OpinionQA'
):
    """
    Load data from file, preprocess, and save results.
    
    Args:
        input_file: Path to input JSON file containing list of survey instances
        output_file: Path to output JSON file
        wave: Wave number for the survey
        year: Survey year
        min_group_size: Minimum group size threshold for demographic groups
        min_group_size_overall: Minimum group size threshold for overall groups
        dataset_name: Name of the dataset
    """
    # Load data
    with open(input_file, 'r') as f:
        data_list = json.load(f)
    
    # Preprocess
    processed_data = preprocess_survey_data(
        data_list, 
        wave=wave, 
        year=year,
        min_group_size=min_group_size,
        min_group_size_overall=min_group_size_overall,
        dataset_name=dataset_name
    )
    
    # Save results
    with open(output_file, 'w') as f:
        json.dump(processed_data, f, indent=2)
    
    print(f"Output saved to {output_file}")
    
    return processed_data


def get_response_mapping_report(data_list: List[Dict]) -> Dict:
    """
    Generate a report showing how responses are mapped to possible answers.
    Useful for debugging response normalization.
    
    Args:
        data_list: List of survey instances
        
    Returns:
        Dictionary with mapping information per question
    """
    report = {}
    
    for instance in data_list:
        for response in instance['responses']:
            q_var = response['Question_var']
            if q_var not in report:
                possible_answers = parse_possible_answers(response['Possible_Answers'])
                report[q_var] = {
                    'possible_answers': possible_answers,
                    'responses_found': defaultdict(int),
                    'unmapped_responses': defaultdict(int)
                }
            
            resp_value = response['Response']
            if resp_value:
                normalized = normalize_response(resp_value, report[q_var]['possible_answers'])
                if normalized in report[q_var]['possible_answers']:
                    report[q_var]['responses_found'][normalized] += 1
                else:
                    report[q_var]['unmapped_responses'][resp_value] += 1
    
    # Convert defaultdicts to regular dicts
    for q_var in report:
        report[q_var]['responses_found'] = dict(report[q_var]['responses_found'])
        report[q_var]['unmapped_responses'] = dict(report[q_var]['unmapped_responses'])
    
    return report


def get_demographic_summary(data_list: List[Dict]) -> Dict[str, Dict[str, int]]:
    """Get count of each demographic value in the data (only for grouping demographics)."""
    demo_counts = defaultdict(lambda: defaultdict(int))
    for instance in data_list:
        for key in DEMOGRAPHIC_KEYS:
            value = instance['feature'].get(key, '')
            demo_counts[key][value] += 1
    return {k: dict(v) for k, v in demo_counts.items()}


def validate_demographics(data_list: List[Dict]) -> Dict[str, List[str]]:
    """Check for any demographic values not in the predefined list."""
    invalid_values = defaultdict(list)
    for instance in data_list:
        for key in DEMOGRAPHIC_KEYS:
            value = instance['feature'].get(key)
            if value is not None and key in DEMOGRAPHIC_VALUES:
                if value not in DEMOGRAPHIC_VALUES[key]:
                    if value not in invalid_values[key]:
                        invalid_values[key].append(value)
    return dict(invalid_values)



In [ ]:
processed = preprocess_survey_data(
    processed_gss, 
    wave=0.0, 
    year=2024, 
    min_group_size=100,
    dataset_name='GSS'
)


In [ ]:
with open('./dataset/gss/gss_grouped.json', 'w') as f:
    json.dump(processed, f)

In [ ]:
processed[0]